In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [9]:
cars = pd.read_csv("cars.csv")

cars = cars.sample(frac=1).reset_index(drop=True)  # shuffle


In [10]:
split_ratio = 0.7
split_index = int(len(cars) * split_ratio)
test, train = cars[split_index:], cars[:split_index]

In [11]:
test_features, test_labels = test[['displacement','horsepower','weight','acceleration']].values, test['mpg'].values.reshape(-1,1)
train_features, train_labels = train[['displacement','horsepower','weight','acceleration']].values, train['mpg'].values.reshape(-1,1)

train_mean = train_features.mean(axis=0)
train_std = train_features.std(axis=0)

train_features = (train_features - train_mean) / train_std

bias = np.ones((train_features.shape[0],1))

train_features = np.hstack((bias, train_features))

test_features = (test_features - train_mean) / train_std

bias = np.ones((test_features.shape[0],1))

test_features = np.hstack((bias, test_features))

def to_one_hot(X):
    classes = np.where(X[:,0] < 15, 0,
              np.where(X[:,0] < 30, 1, 2))
    return np.eye(3)[classes]

test_labels = to_one_hot(test_labels) 
train_labels = to_one_hot(train_labels) 



In [12]:
import numpy as np
import matplotlib.pyplot as plt

class LogisticRegression:
    def __init__(self, learning_rate=0.001):
        self.learning_rate = learning_rate
        self.weight = None
        self.cross_entropy_history = []
        self.mean = None
        self.std = None

    # Softmax for multi-class prediction
    def softmax(self, X):
        X = X - np.max(X, axis=1, keepdims=True)  # row-wise stability
        expX = np.exp(X)
        return expX / np.sum(expX, axis=1, keepdims=True)

    # Cross-entropy loss
    def cross_entropy(self, features, labels):
        preds = self.softmax(features @ self.weight)
        preds = np.clip(preds, 1e-15, 1 - 1e-15)
        return -np.mean(np.sum(labels * np.log(preds), axis=1))

    # Gradient update
    def gradient_descent(self, features, labels):
        preds = self.softmax(features @ self.weight)
        gradients = features.T @ (preds - labels) / features.shape[0]
        self.weight -= self.learning_rate * gradients

    # Fit model with mini-batch gradient descent
    # Assumes bias column is already included in X
    def fit(self, X, y, batch_size=32, epochs=100):
        n_samples, n_features = X.shape
        n_classes = y.shape[1]

        # Standardize features (exclude bias column)
        self.mean = X[:, 1:].mean(axis=0)
        self.std = X[:, 1:].std(axis=0)
        X[:, 1:] = (X[:, 1:] - self.mean) / self.std

        # Initialize weights
        self.weight = np.zeros((n_features, n_classes))

        for _ in range(epochs):
            idx = np.random.permutation(n_samples)
            X_shuffled = X[idx]
            y_shuffled = y[idx]

            for start in range(0, n_samples, batch_size):
                end = min(start + batch_size, n_samples)
                self.gradient_descent(X_shuffled[start:end], y_shuffled[start:end])

            ce = self.cross_entropy(X, y)
            self.cross_entropy_history.append(ce)

            # Dynamic learning rate adjustment
            if len(self.cross_entropy_history) > 1:
                if ce > self.cross_entropy_history[-2]:
                    self.learning_rate /= 2
                else:
                    self.learning_rate *= 1.01

    # Predict probabilities
    def predict_proba(self, X):
        X_scaled = X.copy()
        X_scaled[:, 1:] = (X_scaled[:, 1:] - self.mean) / self.std
        return self.softmax(X_scaled @ self.weight)

    # Predict class labels
    def predict(self, X):
        probs = self.predict_proba(X)
        return np.argmax(probs, axis=1)

    # Evaluate accuracy
    def test(self, X, y):
        preds = self.predict(X)
        actual = np.argmax(y, axis=1)
        accuracy = (preds == actual).mean()
        print(f"Accuracy: {accuracy:.4f}")
        return accuracy

    # Plot training loss
    def plot_loss(self):
        plt.plot(self.cross_entropy_history)
        plt.xlabel("Epochs")
        plt.ylabel("Cross-Entropy Loss")
        plt.title("Training Loss")
        plt.show()

In [ ]:
# Fit the model
model = LogisticRegression(learning_rate=0.001)
model.fit(X=train_features, y=train_labels, batch_size=3, epochs=100)

# Predict probabilities and class for a new sample
sample = np.array([[1, 307, 130, 1.753, 122]])  # bias included as first column
probs = model.predict_proba(sample)
pred_class = model.predict(sample)

print("Predicted probabilities:", probs)
print("Predicted class:", pred_class)

Predicted probabilities: [[1.00000000e+000 1.58005714e-073 2.60978454e-201]]
Predicted class: [0]


In [14]:
model.test(X=test_features,y=test_labels)

Accuracy: 0.7881


np.float64(0.788135593220339)